<div style="background:#12314a;color:#ffffff;padding:26px 30px;border-radius:10px;border-left:7px solid #22a6a1;">
  <div style="font-size:12px;letter-spacing:2px;color:#9ee3df;">MURSHID | TECHNICAL ASSIGNMENTS</div>
  <h1 style="color:#ffffff;margin:12px 0 8px 0;font-size:30px;">Day 02 — Local Model Evaluation</h1>
  <p style="color:#d9e8f2;font-size:15px;margin:0">Three-question test set, keyword proxy, and inference measurements</p>
</div>

**Author:** Setah Saad  
**Training program:** LLM Application Engineering · SDA-AIE-213  
**Institution:** [SDAIA Academy](https://github.com/SDAIAAcademy)  
**Notebook type:** Independent practical homework  
**Documentation language:** English

---

<div style="border-bottom:2px solid #087f83;padding-bottom:5px;margin-top:20px"><h2 style="color:#12314a;margin-bottom:0">01 | Assignment overview</h2></div>

**Purpose.** Evaluate a local pretrained model against three related public-service questions using a deliberately limited measurable keyword proxy.

**Learning objectives**

- Run the same pretrained model on a defined question set.
- Measure inference duration and input/output token counts.
- Check whether each answer contains both “reference” and “number”.
- Export per-case records for manual assessment.

**Technical environment**

| Component | Usage |
|:--|:--|
| Runtime | Python 3 · Google Colab |
| Model | google/flan-t5-small on CPU |
| Test data | Three synthetic question variants |
| Metric | reference + number keyword match (not semantic accuracy) |
| Output | murshid_live_outputs/evaluation_result.json |

**Execution flow**

`Prepare environment → Load model → Run 3 questions → Measure and score → Review generated answers → Export JSON`

<div style="border-left:4px solid #087f83;background:#eef7f7;padding:12px 16px;margin:12px 0"><b>Scope and limitations</b><br>The keyword check is an intentionally weak proxy. A match does not guarantee a correct answer, and an answer can be correct without using those exact words. The notebook performs a real local inference run, but it does not claim formal model quality or security assurance.</div>


<div style="border-bottom:2px solid #087f83;padding-bottom:5px;margin-top:20px"><h2 style="color:#12314a;margin-bottom:0">02 | Dependency and runtime checks</h2></div>

Install the model dependencies and display runtime/library details. The setup uses `pip install -U`, so library versions can differ across runs.

**Expected observation:** Installed package versions and selected device. The evaluation function below explicitly moves its model to CPU.


In [ ]:
# Install required libraries
%pip install -U transformers sentencepiece

# Import libraries
import sys
import torch
import transformers
import sentencepiece

# Check Python version
print("Python version:", sys.version)

# Check library versions
print("Torch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("SentencePiece version:", sentencepiece.__version__)

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Installation status
print("All libraries installed successfully!")
print("Environment is ready!")






   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 57.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 18.1 MB/s  0:00:00
  Attempting uninstall: sentencepiece
    Found existing installation: sentencepiece 0.2.2
    Uninstalling sentencepiece-0.2.2:
      Successfully uninstalled sentencepiece-0.2.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers]
Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Torch version: 2.14.1+cpu
Transformers version: 5.23.0
SentencePiece version: 0.2.3
Device: cpu
All libraries installed successfully!
Environment is ready!


<div style="border-bottom:2px solid #087f83;padding-bottom:5px;margin-top:20px"><h2 style="color:#12314a;margin-bottom:0">03 | Reserved cell</h2></div>

This original code cell is empty. It is retained to preserve the source notebook layout; no execution is required here.


<div style="border-bottom:2px solid #087f83;padding-bottom:5px;margin-top:20px"><h2 style="color:#12314a;margin-bottom:0">04 | Evaluation function and metric definition</h2></div>

The function downloads/loads a specific model revision, defines three semantically similar questions, and generates a response for each with deterministic decoding (`do_sample=False`, `num_beams=1`).

| Recorded field | Interpretation |
|:--|:--|
| `reference_keyword_hit` | Whether both target terms appear in the generated answer |
| `inference_seconds` | Per-question generation time on the current runtime |
| `input_tokens` | Prompt token count |
| `output_tokens_including_special` | Generated-token count, including special tokens |
| `manual_review` | Reminder to assess meaning and unsupported claims |

The summary contains `keyword_hits` and `total`, with the raw cases saved to JSON. **Do not interpret keyword hits as semantic accuracy.**


In [ ]:
# Independent real-model evaluation homework solution. Same pinned dependencies as demo.
import json
import time
from pathlib import Path


def run_evaluation_homework():
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    torch.set_num_threads(2)
    model_id = 'google/flan-t5-small'
    revision = '0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab'
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision, trust_remote_code=False)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id, revision=revision,
        trust_remote_code=False, use_safetensors=True).to('cpu').eval()
    context = 'You can check your application status using its reference number.'
    questions = ['How can I check my application status?',
                 'What do I need to check my application status?',
                 'How can I find out the status of my application?']
    rows = []
    for case_number, question in enumerate(questions, 1):
        prompt = f'Answer the question using the context.\nContext: {context}\nQuestion: {question}\nAnswer:'
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=256)
        start = time.perf_counter()
        with torch.inference_mode():
            tokens = model.generate(**inputs, max_new_tokens=48, do_sample=False, num_beams=1)
        answer = tokenizer.decode(tokens[0], skip_special_tokens=True).strip()
        assert answer, 'Empty generation; inspect runtime.'
        # A deliberately weak measurable proxy, not a semantic judge.
        keyword_hit = 'reference' in answer.lower() and 'number' in answer.lower()
        rows.append({'id': f'E{case_number:02d}', 'question': question, 'answer': answer, 'reference_keyword_hit': keyword_hit,
                     'input_tokens': int(inputs['input_ids'].shape[-1]),
                     'output_tokens_including_special': int(tokens.shape[-1]),
                     'inference_seconds': round(time.perf_counter() - start, 3)})
    result = {'mode': 'real_local_model', 'model': model_id, 'model_revision': revision,
              'generation': {'do_sample': False, 'num_beams': 1, 'max_new_tokens': 48},
              'metric': 'reference+number keyword proxy; NOT semantic accuracy',
              'keyword_hits': sum(x['reference_keyword_hit'] for x in rows), 'total': len(rows),
              'external_inference_api_calls': 0, 'cases': rows,
              'manual_review': 'Check every answer for correct meaning and unsupported facts. Record failures; do not edit expected criteria to force a pass.'}
    out = Path('murshid_live_outputs'); out.mkdir(exist_ok=True)
    (out / 'evaluation_result.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    print(json.dumps(result, indent=2))
    return result





<div style="border-bottom:2px solid #087f83;padding-bottom:5px;margin-top:20px"><h2 style="color:#12314a;margin-bottom:0">05 | Execute and inspect results</h2></div>

Call `run_evaluation_homework()` to generate the responses and output the evaluation JSON. The returned dictionary is also assigned to `result` for subsequent inspection.

**Expected output:** A JSON object containing three cases, keyword-hit totals, token counts, latency, and a manual-review note. Actual scores and timing depend on execution and must be read from the run output rather than assumed.


In [ ]:
result = run_evaluation_homework()

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

{
  "mode": "real_local_model",
  "model": "google/flan-t5-small",
  "model_revision": "0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab",
  "generation": {
    "do_sample": false,
    "num_beams": 1,
    "max_new_tokens": 48
  },
  "metric": "reference+number keyword proxy; NOT semantic accuracy",
  "keyword_hits": 3,
  "total": 3,
  "external_inference_api_calls": 0,
  "cases": [
    {
      "id": "E01",
      "question": "How can I check my application status?",
      "answer": "Use its reference number",
      "reference_keyword_hit": true,
      "input_tokens": 34,
      "output_tokens_including_special": 6,
      "inference_seconds": 0.52
    },
    {
      "id": "E02",
      "question": "What do I need to check my application status?",
      "answer": "Reference number",
      "reference_keyword_hit": true,
      "input_tokens": 36,
      "output_tokens_including_special": 4,
      "inference_seconds": 0.143
    },
    {
      "id": "E03",
      "question": "How can I find out the statu

<div style="border-bottom:2px solid #087f83;padding-bottom:5px;margin-top:20px"><h2 style="color:#12314a;margin-bottom:0">90 | Technical review and references</h2></div>

### Engineering takeaways
- Separates deterministic inference configuration from evaluation criteria.
- Collects per-case evidence rather than reporting one unexplained score.
- Keeps manual review essential when automated metrics are simplistic.

### Validation boundaries
Only three prompts are used; the test does not assess broad accuracy, fairness, robustness, or security. The notebook output may contain model-download logs in addition to the JSON result.

### Further reading
- [Transformers documentation](https://huggingface.co/docs/transformers/index)
- [FLAN-T5 model card](https://huggingface.co/google/flan-t5-small)
- [Hugging Face evaluation guidance](https://huggingface.co/docs/evaluate/index)
- [Python time documentation](https://docs.python.org/3/library/time.html)

---
*This notebook documents an educational implementation. Any reported results should be interpreted using the actual executed outputs and the limitations stated above.*
